This notebook estimates domestic production domestically consumed for many goods, based on the production data of PRODCOM. The difference between this data source and the others is that this data source only pertains to European countries and that it does not systematically cover data for all European countries. As a result, the country coverage of this data source is not complete. Data points from this data source will be used for commodity/country combinations, but for missing data points, the default approximation with exiobase will be used.

In [1]:
import pandas as pd
import sqlite3
import country_converter as coco
import json

Connect to the two databases

In [55]:
conn = sqlite3.connect('C://Users/11max/PycharmProjects/Regioinvent/trade_data/v5/trade_data.db')

full_baci_conn = sqlite3.connect('C://Users/11max/PycharmProjects/Regioinvent/trade_data/v5/baci_trade_data_full_version.db')

In [4]:
# load PRODCOM data
prodcom = pd.read_csv('estat_ds-059359_en.csv', low_memory=False)
# keep interesting data only
prodcom = prodcom.loc[prodcom.indicators == 'APRODQNT']
# truncate the classification to 6 digits. 8 digits is not stable for our work.
prodcom.loc[:, 'product'] = [i[:-2] for i in prodcom.loc[:, 'product']]
# keep useful columns
prodcom = prodcom.loc[:, ['reporter', 'product', 'TIME_PERIOD', 'OBS_VALUE']]
# change values in float (and not strings)
prodcom.loc[:, 'OBS_VALUE'] = prodcom.loc[:, 'OBS_VALUE'].astype(float)
# change PRODCOM data from kg to tonnes (to match with BACI)
prodcom.loc[:, 'OBS_VALUE'] /= 1000
# groupby the different former 8 digit code to the 6 digits level
prodcom = prodcom.groupby(['reporter', 'product', 'TIME_PERIOD']).sum().reset_index()

In [5]:
# convert country codes to 3-letters (to match with BACI)
with open('C://Users/11max/PycharmProjects/Regioinvent/src/regioinvent/data/Regionalization/ei3.12/COMTRADE_to_ecoinvent_geographies.json', 'r') as f:
    two_to_three_country_codes = json.load(f)

two_to_three_country_codes = {v:k for k, v in two_to_three_country_codes.items()}
prodcom.reporter = [two_to_three_country_codes[i] if i in two_to_three_country_codes.keys() else 'to_drop' for i in prodcom.reporter]
prodcom = prodcom.loc[prodcom.reporter != 'to_drop']

In [6]:
# load concordance between PRODCOM and BACI
prodcom_concordance = pd.read_excel('PRODCOM_to_regioinvent.xlsx', dtype='str', index_col=[0]).drop('Name', axis=1).dropna(axis=0, how='all')

In [7]:
# only keep prodcom commodities for which we have found concordances or simply remove the ones we do not need
prodcom = prodcom.loc[prodcom.loc[:,'product'].isin(prodcom_concordance.index)]

In [8]:
# extract relevant export values from BACI
required_hs_export_values = set(prodcom_concordance.stack().tolist())

placeholders = ', '.join([f"'{str(val)}'" for val in required_hs_export_values])

query = f"SELECT * FROM [International trade data] WHERE cmdCode IN ({placeholders})"
data = pd.read_sql(query, full_baci_conn)

In [9]:
# determine net exports
import_data = data.set_index(['cmdCode', 'refYear', 'importer', 'exporter']).drop(
    ['value (1,000 USD)'], axis=1).groupby(['cmdCode', 'refYear', 'importer']).sum().reset_index()
export_data = data.set_index(['cmdCode', 'refYear', 'exporter']).drop(
    ['value (1,000 USD)', 'importer'], axis=1).groupby(['cmdCode', 'refYear', 'exporter']).sum().reset_index()
import_data = import_data.rename(columns={'importer': 'country', 'quantity (t)': 'import_qty'})
export_data = export_data.rename(columns={'exporter': 'country', 'quantity (t)': 'export_qty'})

# outer merge to ensure we don't lose data 
net_exports_data = pd.merge(
    export_data, 
    import_data, 
    on=['cmdCode', 'refYear', 'country'], 
    how='outer'
)

# fill NaNs with 0 (so subtraction works if one side is missing)
net_exports_data[['export_qty', 'import_qty']] = net_exports_data[['export_qty', 'import_qty']].fillna(0)

# calculate Net Exports
net_exports_data['net_export'] = net_exports_data['export_qty'] - net_exports_data['import_qty']

net_exports_data = net_exports_data[['cmdCode', 'refYear', 'country', 'net_export']]

In [10]:
# remove negative net export balance
net_exports_data = net_exports_data[net_exports_data.loc[:,'net_export'] > 0].dropna()

In [11]:
# Convert the concordance Series/DF into a clean mapping DataFrame
mapping = prodcom_concordance.stack().reset_index()
mapping.columns = ['prodcom_code', 'level_1', 'cmdCode']
mapping = mapping[['prodcom_code', 'cmdCode']]

# Ensure types match for merging (e.g., both strings or both ints)
mapping['cmdCode'] = mapping['cmdCode'].astype(str)
net_exports_data['cmdCode'] = net_exports_data['cmdCode'].astype(str)
prodcom['product'] = prodcom['product'].astype(str)

# Join net_exports with the mapping to see which PRODCOM each HS belongs to
hs_with_prodcom = net_exports_data.merge(mapping, on='cmdCode', how='inner')

In [12]:
# Group by PRODCOM, Year, and Country
hs_aggregated = hs_with_prodcom.groupby(['prodcom_code', 'refYear', 'country'])['net_export'].sum().reset_index()

# Merge with the PRODCOM production data
# Note: aligning 'refYear' with 'TIME_PERIOD' and 'country' with 'reporter'
combined_prod = prodcom.merge(
    hs_aggregated, 
    left_on=['product', 'TIME_PERIOD', 'reporter'],
    right_on=['prodcom_code', 'refYear', 'country'],
    how='left'
).fillna(0)

# Calculate the "Domestic Production" (Production - Net Exports)
combined_prod['adjusted_production'] = combined_prod['OBS_VALUE'] - combined_prod['net_export']

C:\Users\11max\AppData\Local\Temp\ipykernel_14652\2327717834.py:11: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ).fillna(0)


In [33]:
# 1. Get total exports per PRODCOM group (for the ratio denominator)
# Note: Using absolute export values often makes more sense for ratios than net exports
total_exports_per_group = hs_with_prodcom.groupby(['prodcom_code', 'refYear', 'country'])['net_export'].transform('sum')

# 2. Calculate the ratio for each HS code
# Handle division by zero in case a group has 0 total exports
hs_with_prodcom['distribution_ratio'] = hs_with_prodcom['net_export'] / total_exports_per_group
hs_with_prodcom['distribution_ratio'] = hs_with_prodcom['distribution_ratio'].fillna(0)

# 3. Merge the adjusted production back into the HS-level dataframe
final_distribution = hs_with_prodcom.merge(
    combined_prod[['prodcom_code', 'refYear', 'country', 'adjusted_production']],
    on=['prodcom_code', 'refYear', 'country'],
    how='left'
)

# 4. Final step: Attribute production based on the ratio
final_distribution['attributed_production'] = (
    final_distribution['adjusted_production'] * final_distribution['distribution_ratio']
)

# Clean up output
final_output = final_distribution[['country', 'refYear', 'prodcom_code', 'cmdCode', 'attributed_production']]

In [14]:
# Identify pairs with non-zero production in PRODCOM
# We filter where OBS_VALUE > 0
active_production_mask = (final_distribution['adjusted_production'] + final_distribution['net_export']) > 0

# If you want to be more direct and use the original OBS_VALUE column 
# (assuming you kept it during the merge):
# active_production_mask = final_distribution['OBS_VALUE'] > 0

# Apply the filter
active_pairs = final_distribution[active_production_mask].copy()

# Select and deduplicate the specific identifying columns
# This gives you the unique Country/Year/Commodity list
active_list = active_pairs[['country', 'refYear', 'prodcom_code', 'cmdCode']].drop_duplicates()

In [34]:
# only keep residual production values for which PRODCOM initially provided data
final_output = final_output.loc[active_list.index]

In [35]:
# in this context, if values are negative, that means that the country truly exported more than its PRODCOM production data -> we will assume that the country does not have production for domestic consumption
final_output.loc[final_output['attributed_production'] < 0, 'attributed_production'] = 0

In [43]:
# for aggregated HS codes (4 digits instead of 6) we recreated the production data
with open('C://Users/11max/PycharmProjects/Regioinvent/src/regioinvent/data/Regionalization/ei3.12/ecoinvent_to_HS.json', 'r') as f:
    codes = json.load(f)

four_digit_codes = [j for j in list(set([i[:-2] for i in final_output.cmdCode])) if j in codes.values()]

for code in four_digit_codes:
    df = final_output[final_output.cmdCode.str.startswith('3103')].drop(['prodcom_code', 'cmdCode'], axis=1).groupby(['country', 'refYear']).sum().reset_index()
    df.loc[:, 'cmdCode'] = code
    final_output = pd.concat([final_output, df])

In [53]:
# indicate that through PRODCOM, we do not cover all countries, but only some European countries
final_output.loc[:, 'country_coverage'] = 'partial'
final_output.loc[:, 'source'] = 'Estimated from PRODCOM data, "Total production" table [ds-059359] @ https://ec.europa.eu/eurostat/databrowser/bulk. Last update of the PRODCOM data: 27/04/2026.'

Add the data to the SQL database inside the "Real production" data table